base code

In [1]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# model_name = "google/flan-t5-small"

# # Load tokenizer + model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# # Prompt (important!)
# prompt = """Classify sentiment as Positive or Negative.
# Text: I love this product.
# Answer:"""

# # Tokenize
# inputs = tokenizer(prompt, return_tensors="pt")

# # Generate output
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=5,
#     do_sample=False
# )

# # Decode result
# result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# print(result)

Initialization

In [ ]:
# import libraries
from sklearn.metrics import f1_score
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
warnings.filterwarnings("ignore")

# set config
results_df = pd.DataFrame(columns=["prompt_template", "accuracy", "F1_score"])
model_name = "google/flan-t5-small"


In [109]:
# Load the dataset
train = pd.read_csv('data/6/train.csv')
train.drop(columns=['data_id'], inplace=True)
train.dropna(inplace=True)

In [110]:
train_df, samples_df = train.iloc[:4000].copy(), train.iloc[4000:].copy()
train_df.shape, samples_df.shape

((4000, 2), (912, 2))

In [23]:
# data cleaning
def clean_text(df, text_col="text", target_col="sentiment"):

    df = df.dropna(subset=[text_col])
    df[text_col] = df[text_col].str.strip()
    df[text_col] = df[text_col].str[:2048]

    df[target_col] = df[target_col].str.lower()
    return df

# inference
def batch_predict(texts, prompt_template, tokenizer, model):
    prompts = [prompt_template.format(t=t) for t in texts]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

def pipeline(df, prompt_template):
    # data cleaning
    df = clean_text(df)

    # Load model
    model_name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Set model to eval mode (faster)
    model.eval()

    # Apply in batches
    batch_size = 32
    preds = []

    for i in range(0, len(df), batch_size):
        batch_texts = df["text"].iloc[i:i+batch_size].tolist()
        preds.extend(batch_predict(batch_texts, prompt_template, tokenizer, model))

    df["predicted"] = [p.strip() for p in preds]

    # evaluation
    df['predicted'] = df['predicted'].str.lower()
    df['sentiment'] = df['sentiment'].str.lower()
    df['correct'] = df['sentiment'] == df['predicted']

    # print('Prompt Template:')
    # print('-'*10)
    # print(prompt_template)
    # print('-'*10)

    accuracy = df['correct'].mean()
    # print(f"Accuracy: {accuracy:.2%}")

    f1 = f1_score(df['sentiment'], df['predicted'], average='weighted')
    print(f"F1-Score: {f1}")

    print('-'*50)

    results = {
        "prompt_template": prompt_template,
        "accuracy": accuracy,
        "F1_score": f1
    }

    return results, df

Baseline

In [5]:
prompt_template = f"Classify sentiment as Positive or Negative.\nText: {{t}}\nAnswer:"
results, df = pipeline(valid_df, prompt_template)
results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
results_df

Prompt Template:
----------
Classify sentiment as Positive or Negative.
Text: {t}
Answer:
----------
Accuracy: 87.14%
F1-Score: 0.8792653061224489
--------------------------------------------------


/var/folders/xq/dznpw7s50vdg77ddlcz18rvw0000gn/T/ipykernel_25920/993365498.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)


,prompt_template,accuracy,F1_score
0,Classify sentiment as Positive or Negative.\nT...,0.871429,0.879265


Stronger zero-shot prompt

In [7]:
zero_shot_prompt = """You are a sentiment classifier.
Classify the sentiment of the text as Positive or Negative.
Answer with only one word: Positive or Negative.

Text: {t}
Answer:"""
results, df = pipeline(valid_df, zero_shot_prompt)


Prompt Template:
----------
You are a sentiment classifier.
Classify the sentiment of the text as Positive or Negative.
Answer with only one word: Positive or Negative.

Text: {t}
Answer:
----------
Accuracy: 87.71%
F1-Score: 0.8839499189056447
--------------------------------------------------


Few-shot prompt

In [14]:
# Few-shot prompt with 3 examples
train_df = pd.read_csv('data/6/train.csv')
train_examples = train_df.sample(5, random_state=2501676)
# Distribution of labels in validation set should be similar to examples
valid_df['sentiment'].value_counts(normalize=True)

sentiment
positive    0.795714
negative    0.204286
Name: proportion, dtype: float64

In [23]:
positive_sample = train_df[train_df['sentiment'] == 'positive'].sample(2, random_state=2501676)
clean_povitive_examples = clean_text(positive_sample)

negative_sample = train_df[train_df['sentiment'] == 'negative'].sample(1, random_state=2501676)
clean_negative_examples = clean_text(negative_sample)

print("Positive Examples:")
print(clean_povitive_examples['text'].tolist()[0])
print(clean_povitive_examples['text'].tolist()[1])
print("\nNegative Example:")
print(clean_negative_examples['text'].tolist()[0])

Positive Examples:
Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
This item worked perfectly for my refrigerator.

Negative Example:
Bought it June 21, died last night. No longevity. Worked good other then that.


In [25]:
one_shot_prompt = """You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(valid_df, one_shot_prompt)
results_df = pd.concat([results_df, pd.DataFrame([results_one_shot])], ignore_index=True)

Prompt Template:
----------
You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: {t}
Answer:
----------
Accuracy: 90.14%
F1-Score: 0.9072445243901734
--------------------------------------------------


In [26]:
two_shot_prompt = """You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: Negative

Text: {t}
Answer:"""

results_two_shot, df = pipeline(valid_df, two_shot_prompt)
results_df = pd.concat([results_df, pd.DataFrame([results_two_shot])], ignore_index=True)

Prompt Template:
----------
You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: Negative

Text: {t}
Answer:
----------
Accuracy: 86.71%
F1-Score: 0.8772892986578518
--------------------------------------------------


In [24]:
few_shot_prompt = """You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: This item worked perfectly for my refrigerator.
Answer: Positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: Negative

Text: {t}
Answer:"""

results_few, df = pipeline(valid_df, few_shot_prompt)
results_df = pd.concat([results_df, pd.DataFrame([results_few])], ignore_index=True)

Prompt Template:
----------
You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: This item worked perfectly for my refrigerator.
Answer: Positive

Text: Bought it June 21, died last night. No longevity. Worked good other then that.
Answer: Negative

Text: {t}
Answer:
----------
Accuracy: 86.29%
F1-Score: 0.873510633243753
--------------------------------------------------


Label-definition prompt

In [30]:
lable_definition_prompt = """Classify the sentiment of the text.

Positive = happy, satisfied, favorable
Negative = unhappy, dissatisfied, unfavorable

Answer with only one word: Positive or Negative.

Text: {t}
Answer:"""
results_lable_definition, df = pipeline(valid_df, lable_definition_prompt)
results_df = pd.concat([results_df, pd.DataFrame([results_lable_definition])], ignore_index=True)

Prompt Template:
----------
Classify the sentiment of the text.

Positive = happy, satisfied, favorable
Negative = unhappy, dissatisfied, unfavorable

Answer with only one word: Positive or Negative.

Text: {t}
Answer:
----------
Accuracy: 85.71%
F1-Score: 0.8670711527854384
--------------------------------------------------


Different Examples in one shot

In [27]:
for exp in train_df['text'].tail(10):
    one_shot_prompt = f"""You are a sentiment classifier.
    Answer with only one word: Positive or Negative.

    Text: {exp}
    Answer: Positive

    """+"""Text: {t}
    Answer:"""
    print(exp)

    results_one_shot, df = pipeline(train_df, one_shot_prompt)


With a second person to help tilt the machine back, all four were quickly swapped-out from the bottom.  I was amazed at how quickly it was done, but they've made just the difference we wanted!  Decent steel and plastic construction, so I'm hoping these last at least as long as the OE ones.
F1-Score: 0.9160581689985677
--------------------------------------------------
It worked out perfectly
F1-Score: 0.9075777943739108
--------------------------------------------------
Don't bother... Within a year flashs an EEO code and no longer functions. Customer service is a terminal hold.
F1-Score: 0.9082042660502259
--------------------------------------------------
They were affordable and easy to install- look better than the original.
F1-Score: 0.9167043607066729
--------------------------------------------------
It was an exact fit at a good price.
F1-Score: 0.9075777943739108
--------------------------------------------------
Plug'n and play!!!
F1-Score: 0.9121290322580644
----------------

In [30]:
one_shot_prompt = """You are a sentiment classifier.
Answer with only one word: Positive or Negative.

Text: Love this machine! Makes great ice quickly. I bag it each time it fills up and use it at a later time. Buying a 2nd one for our camper!
Answer: Positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.9037809651244877
--------------------------------------------------


Prompt paraphrasing

In [59]:
samples_df[samples_df['sentiment'] == 'negative']['text'].tolist()[:5]

['this item did not work',
 "This thing works pretty good but it's extremely loud. I have it in my office at work and sometimes it get annoying its so loud. It's pretty cheap so I would recommend putting it in a shop but not in an office.",
 'The part came quickly. The only problem is that it is not a Powerburner knob as advertised, it is an Accusimmer knob.',
 'Didn’t work',
 'Don’t know how the product worked, it arrived Witt the front door completely damaged. Returning and asking for an exchange. Will update after receiving the new one.']

Baseline on 4000 samples is 0.9

In [53]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive, negative

Answer with exactly one word: positive or negative. Do not explain.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.9080350387430433
--------------------------------------------------


Baseline on 1000 samples is 0.9

In [111]:
train_df = train_df.tail(1000)

In [60]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive, negative

Answer with exactly one word: positive or negative. Do not explain.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.9030272261506542
--------------------------------------------------


Baseline using shorter example didnt work. f1 was 0.89

In [ ]:
# one_shot_prompt = """Classify the sentiment of the text.

# Labels: positive, negative

# Answer with exactly one word: positive or negative. Do not explain.

# Text: It works great and does not have the waste from the pods.
# Answer: positive

# Text: {t}
# Answer:"""

# results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.8916766557385777
--------------------------------------------------


Error Analysis using LLM

In [62]:
df[df['correct'] == False][['text', 'sentiment', 'predicted']].to_csv('data/gpt_misclassified.csv', index=False)

In [64]:
misclassified_df = df[df['correct'] == False][['text', 'sentiment', 'predicted']]
misclassified_df['sentiment'].value_counts()

sentiment
positive    79
negative    23
Name: count, dtype: int64

In [ ]:
misclassified_df[misclassified_df['sentiment'] == 'positive']['text']

2           Looks good but did not need.  Put into stock
7      The factory filter is very expensive. This doe...
16     I got 6 on here for less than half of what Sea...
33     They work as well as any other coffee filter. ...
61     Purchased a new Maytag dishwasher on Black Fri...
                             ...                        
968    I bought the 7cube model, came with a quarter ...
969    I love the self sealing ends .I have  a new mo...
974    Replaced several parts on gas dryer. Turned ou...
982                                         Oven repair.
990    The burner coil itself is of good quality but ...
Name: text, Length: 79, dtype: object

Add negative example failed -> 0.87

In [ ]:
one_shot_prompt = """Classify the sentiment of the following text as either positive or negative.

If the text contains both positive and negative statements, choose the label that best matches the overall sentiment.  Return exactly one lowercase word (“positive” or “negative”).  Do not explain your answer.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Sentiment: positive

Example:
Text: This thing works pretty good but it's extremely loud. I have it in my office at work and sometimes it get annoying its so loud. It's pretty cheap so I would recommend putting it in a shop but not in an office.
Sentiment: negative

Now classify:
Text: {t}
Sentiment:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.8724487190312937
--------------------------------------------------


In [70]:
one_shot_prompt = """Classify the sentiment of the following text as either positive or negative.

If the text contains both positive and negative statements, choose the label that best matches the overall sentiment.  Return exactly one lowercase word (“positive” or “negative”).  Do not explain your answer.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Sentiment: positive

Now classify:
Text: {t}
Sentiment:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.8750158415085697
--------------------------------------------------


Add guide about handling but, however, etc -> Improved: 0.91

In [112]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive, negative

Answer with exactly one word: positive or negative. Do not explain.
After "but", "however", or "although", the following sentiment is more important.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.9251790860983518
--------------------------------------------------


In [113]:
df[df['correct'] == False][['text', 'sentiment', 'predicted']].to_csv('data/gpt_misclassified.csv', index=False)

In [ ]:
Classify the sentiment of the text.

If the text is unclear, very short, or lacks strong sentiment words, label it as negative.

Labels: positive, negative  
Answer with exactly one word.

Text: {t}
Answer:

## Not in thread

Add: If you were unsure -> positive

In [86]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive, negative

Answer with exactly one word: positive or negative. Do not explain.
After "but", "however", or "although", the following sentiment is more important.
If the text contains both positive and negative statements, or if the sentiment is unclear, choose positive.
Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.9117737477332951
--------------------------------------------------


In [89]:
df[df['correct'] == False][['text', 'sentiment', 'predicted']].to_csv('data/gpt_misclassified.csv', index=False)

Oposite sentiment of phrase before 'but'

In [90]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive, negative

Answer with exactly one word: positive or negative. Do not explain.
If the text contains "but", "however", or "although", the final sentiment is usually the opposite of the sentiment before these words (e.g., a positive phrase followed by "but" should be classified as negative).

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.9158852309350698
--------------------------------------------------


paraphrase of the above prompt

In [106]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive, negative

Answer with exactly one word: positive or negative. Do not explain.
After "but", "however", or "although", the sentiment typically reverses from the earlier part.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(train_df, one_shot_prompt)

F1-Score: 0.9160285254047047
--------------------------------------------------


In [95]:
df[
    (df['predicted'] == 'negative') &
    (df['sentiment'] == 'positive') &
    (df['text'].str.contains(r'\b(but|however|although)\b', case=False, na=False))
]['text'].tolist()

['Looks good but did not need.  Put into stock',
 "They work as well as any other coffee filter.  However the size is hard to find for 4 cup drip pots.  These are the only ones I've found that fit.  A bit expensive but no choice.  As far as I know no others are made.",
 "Works great! But if you're in the minority of people (like me) who didn't consider the stove was higher than the countertop, then the gap cover is not going to stay in place very well.",
 "The parts are good and about half what Whirlpool charges. The fix was fairly quick, but do yourself a favor and find a good YouTube video to walk you through if you aren't sure. The directions they send with the parts are pretty poor.",
 'The part does not fit. I read reviews after purchasing it and it appears it is threaded backwards. Screws on backwards but not when properly oriented. Waste of Money',
 'I thought it was easy to install but, 9 months later it burned out. I do keep my dryer clean inside and out for lint deposits',
 '

In [97]:
train_df['text'].str.lower()

0       best sock ever made. consistent quality...i hi...
1                 it always makes the water taste so good
2            looks good but did not need.  put into stock
3       unit worked perfectly for approximately 6 days...
4       they fit perfectly on my ancient general elect...
                              ...                        
996                  great product and easy installation.
997     this product works better than the one that ca...
998     it fit great and seller even sent us complimen...
999     i wanted this to work so bad! arrived new, fol...
1000     this part arrived quickly and was a perfect fit.
Name: text, Length: 1000, dtype: object

In [99]:
lower_df = train_df.copy() 
lower_df['text'] = lower_df['text'].str.lower()
lower_df

,sentiment,text
0,positive,best sock ever made. consistent quality...i hi...
1,positive,it always makes the water taste so good
2,positive,looks good but did not need. put into stock
3,negative,unit worked perfectly for approximately 6 days...
4,positive,they fit perfectly on my ancient general elect...
...,...,...
996,positive,great product and easy installation.
997,positive,this product works better than the one that ca...
998,positive,it fit great and seller even sent us complimen...
999,negative,"i wanted this to work so bad! arrived new, fol..."


text to lower case

In [101]:
one_shot_prompt = """Classify the sentiment of the text.

Labels: positive, negative

Answer with exactly one word: positive or negative. Do not explain.
After "but", "however", or "although", the sentiment typically reverses from the earlier part.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(lower_df, one_shot_prompt)

F1-Score: 0.9103882975488109
--------------------------------------------------


Dont mention it is sentiment analysis

In [104]:
one_shot_prompt = """Classify this text into positive or negative.

Answer with exactly one word: positive or negative. Do not explain.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(lower_df, one_shot_prompt)

F1-Score: 0.8995203358982214
--------------------------------------------------


Add role

In [105]:
one_shot_prompt = """
You are a sentiment classifier.
Classify this text into positive or negative.

Answer with exactly one word: positive or negative. Do not explain.

Text: These knobs are awesome! The last set we bought weren’t raised like the original ones and all broke off within months. These are so easy to turn, and I love that they are raised so we can see the lights under the knobs again. They seem very sturdy!
Answer: positive

Text: {t}
Answer:"""

results_one_shot, df = pipeline(lower_df, one_shot_prompt)

F1-Score: 0.8995203358982214
--------------------------------------------------
